# W2b — the SOFR pack convexity screen, LIVE

> *"Convexity adjustments for 1y SOFR packs are computed as the spread between
> the pack's rate (the average of 4 SOFR rates in the pack) and
> matched-maturity forward 1y CME swap rate. The model for convexity
> adjustment is the Ho-Lee model calibrated to cap/floor vols. Implied vol is
> calculated by matching the model to the observed convexity adjustment.
> Realized vol is 3m realized vol of the corresponding pack. For each
> valuation metric, we mark three best short convexity trades in bold."*
>
> — Citi Research, *Rates Vol Lab — Forward steepener and vol divergence*,
> 12-Jun-2023, Figure 58 (the SOFR restatement of Citi's own Eurodollar screen)

and the trade the screen selects, verbatim (Citi, 13-Jan-2017 / 09-Feb-2017):

> *"Sell $100k DV01 of Blues convexity adjustment, i.e. buy 1000 of H0-Z0 packs
> (1000 of each of the four contracts) and pay $1bn on a matched-maturity
> (3/18/20-3/17/21) CME swap."*

**This notebook is not a backtest.** It is the screen run on the most recent
data on this machine, with the enrichment attached and the historical context
that decides how much of it to believe. Every number below is measured at
execution time; nothing is transcribed.

The verdict is in section 12, and it is a qualified one: the screen is
information about *where* the convexity adjustment is rich, and the rule it
feeds was measured in the W2b backtest to make approximately nothing net of
costs. Read section 11 before acting on section 7.

In [1]:
import datetime as dt
import json
import math
import os
import pathlib
import sys

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

REPO = pathlib.Path.cwd()
while not (REPO / "RVUtils").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

import RVUtils.ConvexityRV.strat2_sofr_convexity as S2
from RVUtils.ConvexityRV import ca_signals as CS
from RVUtils.ConvexityRV.strat2_q20 import CITI_SOFR_20230609

DATA = REPO / "notebooks" / "data" / "convexity_rv"
pd.set_option("display.width", 260, "display.max_columns", 45)

RUN_AT = dt.datetime.now()
print(f"repo      {REPO}")
print(f"executed  {RUN_AT:%Y-%m-%d %H:%M}")

C:\Users\chris\clee\ARBS-cvx2\RVUtils\ConvexityRV\curve_ops.py:61: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



repo      C:\Users\chris\clee\ARBS-cvx2
executed  2026-08-21 05:51


## 1. The config

Every knob, and the reason for its value. The controlling requirement is
**parity with the W2b backtest**: a screen run with different knobs is not
constrained by that backtest's verdict, and section 11 leans on that verdict
entirely. `w2b_run.json` records what the backtest actually ran, and the cell
below asserts against it rather than trusting this file's comments.

In [2]:
PANEL_PATH = DATA / "strat2_q20_panel.parquet"
panel_all = pd.read_parquet(PANEL_PATH)
panel_all["date"] = pd.to_datetime(panel_all["date"])

GATED = panel_all[panel_all["gate_ok"]].copy()
AS_OF = GATED["date"].max()                 # most recent date the gate passes

CONFIG = S2.Strat2Config(
    start=dt.date(2021, 1, 1),   # W2b's window; z-scores need the run-up before it
    end=AS_OF.date(),            # the live date, not a hardcoded one
    rank_start=2,                # rank 1 has no nearer pack to difference the 3m roll against
    n_packs=16,                  # ranks 2..17, so Reds(5)/Greens(9)/Blues(13)/Golds(17) are all inside
    n_contracts=20,              # SFRCM1..20; depth 20 is what puts Golds in daily reach
)
# everything else is the dataclass default and is printed below, including:
#   top_n_per_metric=3        "we mark three best short convexity trades"
#   min_metrics_flagged=0     always hold the top-ranked pack (Citi rolled Blues->Greens)
#   rebalance_freq="BMS"      the screen is consulted on the 1st business day of each month
#   ca_dv01=100_000           Citi's flagship size
print(json.dumps({k: str(v) for k, v in CONFIG.__dict__.items()}, indent=1))

RUN_JSON = DATA / "w2b_run.json"
if RUN_JSON.exists():
    bt_cfg = json.loads(RUN_JSON.read_text())["config"]
    mismatch = {k: (v, str(getattr(CONFIG, k)))
                for k, v in bt_cfg.items()
                if hasattr(CONFIG, k) and k not in ("end",)
                and str(getattr(CONFIG, k)) != v}
    print(f"\nparity vs w2b_run.json: {len(mismatch)} knob(s) differ -> {mismatch}")
    assert not mismatch, (
        "this screen is NOT the rule the W2b backtest measured; section 11's "
        "verdict does not apply to it")
    print(f"OK: identical to the backtested rule except `end` "
          f"({bt_cfg.get('end')} -> {CONFIG.end}, the live date).")
else:
    print("\nw2b_run.json ABSENT — config parity with the backtest is UNVERIFIED.")

{
 "curve": "USD-SOFR-1D",
 "swap_source": "CITIVELO_EXCEL",
 "futures_source": "BARCHART_STIRF-RL",
 "futures_root": "SR3",
 "n_contracts": "20",
 "rank_start": "2",
 "n_packs": "16",
 "min_priced_contracts": "4",
 "ca_basis_bp": "0.0",
 "round_pack_price_to_tick": "False",
 "holee_convention": "citi",
 "sigma_model_mode": "fit",
 "sigma_fit_degree": "2",
 "sigma_constant_bp": "100.0",
 "ca_floor_bp": "0.1",
 "realized_window_days": "63",
 "realized_annualisation": "252.0",
 "z_window_3m": "63",
 "z_window_1y": "252",
 "week_days": "5",
 "min_history_for_z1y": "252",
 "window_span_tolerance": "0.0",
 "rank_metrics": "('ca_bp', 'ca_z3m', 'ca_z1y', 'vs_model_bp', 'vs_model_z3m', 'vs_model_z1y', 'roll_3m_bp', 'implied_over_realized')",
 "top_n_per_metric": "3",
 "min_metrics_flagged": "0",
 "hedge_enabled": "True",
 "hedge_tenors": "('2Y', '5Y', '10Y')",
 "hedge_regression_days": "252",
 "hedge_min_abs_beta": "1.0",
 "hedge_require_positive_wings": "True",
 "ca_dv01": "100000.0",
 "max_h

## 2. Does the machine give the right answer to a question we already know?

`CITI_SOFR_20230609` freezes **13 published rows x 6 columns** from Citi's
Figure 58 — numbers this code was never fitted to. They are graded here
through the same `daily_screen` the live screen uses.

**This cell deliberately uses a different config from the live screen**:
`rank_start=5, n_packs=13`, which is Citi's own cross-section. The fair-value
model is fitted *across the ranked packs*, so grading it against a table built
on ranks 5–17 requires fitting on ranks 5–17.

The result is a **split**, and the failures are as informative as the passes.

In [3]:
CITI_DATE = dt.date(2023, 6, 9)
cfg_citi = S2.Strat2Config(start=dt.date(2018, 1, 1), end=CITI_DATE,
                           rank_start=5, n_packs=13, n_contracts=20)
_hist = GATED[GATED["date"] <= pd.Timestamp(CITI_DATE)].copy()
_scr_citi = S2.daily_screen(CITI_DATE, _hist, cfg_citi,
                            ts=S2.panel_timeseries(_hist, cfg_citi),
                            model=S2.model_timeseries(_hist, cfg_citi))

_cit = pd.DataFrame(CITI_SOFR_20230609).T
_cit.index.name = "pack"
_j = _scr_citi.join(_cit, how="inner", rsuffix="_citi")
print(f"joined {len(_j)} of {len(_cit)} published rows on {CITI_DATE}")

_side = _j[["rank", "ca_bp", "ca_bp_citi", "ca_model_bp", "model_bp",
            "roll_3m_bp", "roll_3m_bp_citi",
            "implied_vol_bp", "implied_vol_bp_citi",
            "realized_vol_bp", "realized_vol_bp_citi"]]
print("\nours vs Citi's published table, side by side:")
print(_side.astype(float).round(2).to_string())

TIE = []
for ours, theirs in (("ca_bp", "ca_bp_citi"), ("ca_model_bp", "model_bp"),
                     ("vs_model_bp", "vs_model_bp_citi"),
                     ("roll_3m_bp", "roll_3m_bp_citi"),
                     ("implied_vol_bp", "implied_vol_bp_citi"),
                     ("realized_vol_bp", "realized_vol_bp_citi")):
    a = pd.to_numeric(_j[ours], errors="coerce")
    b = pd.to_numeric(_j[theirs], errors="coerce")
    m = a.notna() & b.notna()
    TIE.append({"column": ours, "n": int(m.sum()),
                "pearson": float(a[m].corr(b[m])) if m.sum() > 2 else np.nan,
                "spearman": float(a[m].corr(b[m], method="spearman")) if m.sum() > 2 else np.nan,
                "mean_err_bp": float((a[m] - b[m]).mean()) if m.sum() else np.nan})
TIE = pd.DataFrame(TIE).set_index("column")
print(TIE.round(4).to_string())

assert TIE.at["ca_bp", "n"] == 13, "all 13 published packs must be resolvable"
assert TIE.at["ca_bp", "pearson"] > 0.90, (
    "the measured convexity adjustment must reproduce Citi's published level "
    "in the cross-section; if it does not, nothing downstream is meaningful")
print(f"\nPASSES: ca_bp reproduces at pearson {TIE.at['ca_bp','pearson']:.4f} across "
      f"all 13 rows, and the fitted model at {TIE.at['ca_model_bp','pearson']:.4f}.")
print("FAILS, and why each one is expected rather than a bug:")
print(f"  vs_model_bp   pearson {TIE.at['vs_model_bp','pearson']:+.4f} — the module docstring")
print("                disclaims this column explicitly: our sigma is fitted to the same")
print("                cross-section it is compared against, so vs_model is a residual")
print("                centred on zero. Citi's is a dislocation to an EXTERNAL cap surface.")
_ord = _j.sort_values("rank")
_mono_citi = bool((pd.to_numeric(_ord["ca_bp_citi"]).diff().dropna() > 0).all())
_mono_ours = bool((pd.to_numeric(_ord["ca_bp"]).diff().dropna() > 0).all())
print(f"  roll_3m_bp    pearson {TIE.at['roll_3m_bp','pearson']:+.4f} — a first difference across")
print("                rank, so it amplifies any bumpiness in the CA curve. Citi's CA rises")
print(f"                monotonically in rank that day ({_mono_citi}); ours does not "
      f"({_mono_ours}) —")
print("                see the side-by-side table above, where our CA dips at rank 9.")
_e_ca = pd.to_numeric(_ord["ca_bp"]) - pd.to_numeric(_ord["ca_bp_citi"])
_e_iv = pd.to_numeric(_ord["implied_vol_bp"]) - pd.to_numeric(_ord["implied_vol_bp_citi"])
_err_corr = float(_e_ca.corr(_e_iv))
print(f"  implied_vol   pearson {TIE.at['implied_vol_bp','pearson']:+.4f} — implied vol is the Ho-Lee")
print("                INVERSION of ca_bp, so it cannot be more accurate than ca_bp is.")
print(f"                Its error correlates {_err_corr:+.3f} with the ca_bp error across the 13")
print("                rows: it is the same disagreement passed through the inversion,")
print("                not a second independent failure.")
assert _err_corr > 0.8, (
    "the implied-vol error is NOT explained by the CA error, so it is a second "
    "independent disagreement and the prose above is wrong")
print(f"  realized_vol  only {TIE.at['realized_vol_bp','n']}/13 rows are finite — a 63-day window with")
print("                min_periods=63 returns NaN for any label with a recent gate hole.")
print("                That is the SAME mechanism that shapes the live screen (section 8).")

joined 13 of 13 published rows on 2023-06-09

ours vs Citi's published table, side by side:
       rank  ca_bp  ca_bp_citi  ca_model_bp  model_bp  roll_3m_bp  roll_3m_bp_citi  implied_vol_bp  implied_vol_bp_citi  realized_vol_bp  realized_vol_bp_citi
pack                                                                                                                                                          
M4-H5   5.0   4.04        4.03         4.43      2.94        2.38             0.88          198.65                199.5           217.38                 229.7
U4-M5   6.0   6.89        4.41         5.55      3.79        2.84             0.38          221.35                178.1           196.99                 201.4
Z4-U5   7.0   8.22        5.16         6.68      4.66        1.33             0.74          210.81                167.7           180.08                 180.3
H5-Z5   8.0   8.73        6.10         7.79      5.53        0.51             0.95          192.56               

                  n  pearson  spearman  mean_err_bp
column                                             
ca_bp            13   0.9660    0.9560      -0.0379
ca_model_bp      13   0.9957    1.0000       3.5613
vs_model_bp      13  -0.0877   -0.2088      -3.5977
roll_3m_bp       13  -0.1341   -0.0714       0.0713
implied_vol_bp   13   0.6188    0.6593       4.1923
realized_vol_bp   4   0.9988    1.0000      -3.9946

PASSES: ca_bp reproduces at pearson 0.9660 across all 13 rows, and the fitted model at 0.9957.
FAILS, and why each one is expected rather than a bug:
  vs_model_bp   pearson -0.0877 — the module docstring
                disclaims this column explicitly: our sigma is fitted to the same
                cross-section it is compared against, so vs_model is a residual
                centred on zero. Citi's is a dislocation to an EXTERNAL cap surface.
  roll_3m_bp    pearson -0.1341 — a first difference across
                rank, so it amplifies any bumpiness in the CA curve. Ci

### 2.1 The rounding asymmetry, reproduced literally

From the `daily_screen` docstring, verbatim:

> *"`Implied/Realized` is ROUNDED to 1dp and `Cap vol Impl/Rlzd` is TRUNCATED
> to 1dp — reproduced literally, because that asymmetry is what ties out 26/26
> rows across Citi's ED and SOFR tables."*

It is load-bearing rather than cosmetic: on this date the two rules disagree
on most of the finite rows.

In [4]:
_chk = _scr_citi[["implied_over_realized_raw", "implied_over_realized",
                  "sigma_model_bp", "realized_vol_bp", "capvol_over_realized"]].copy()
_chk["cap_raw"] = _chk["sigma_model_bp"] / _chk["realized_vol_bp"]
_fin = _chk["implied_over_realized_raw"].notna() & _chk["cap_raw"].notna()
_f = _chk[_fin]
print(_f.round(4).to_string())

_want_round = _f["implied_over_realized_raw"].map(lambda v: round(v, 1))
_want_trunc = _f["cap_raw"].map(lambda v: math.trunc(v * 10) / 10)
assert _fin.sum() >= 3, "too few finite rows to grade the rounding rule"
assert (_f["implied_over_realized"] == _want_round).all(), "Implied/Realized is not round-to-1dp"
assert (_f["capvol_over_realized"] == _want_trunc).all(), "Cap vol Impl/Rlzd is not trunc-to-1dp"

_n_differ = int((_f["capvol_over_realized"] != _f["cap_raw"].map(lambda v: round(v, 1))).sum())
print(f"\nOK: round on Implied/Realized, truncate on Cap vol Impl/Rlzd, {_fin.sum()}/13 finite rows.")
print(f"The two rules give a DIFFERENT printed value on {_n_differ} of {int(_fin.sum())} "
      "of them, so the asymmetry is not cosmetic.")

       implied_over_realized_raw  implied_over_realized  sigma_model_bp  realized_vol_bp  capvol_over_realized  cap_raw
pack                                                                                                                   
M4-H5                     0.9138                    0.9        207.9403         217.3768                   0.9   0.9566
U4-M5                     1.1237                    1.1        198.7050         196.9872                   1.0   1.0087
Z4-U5                     1.1707                    1.2        190.0068         180.0773                   1.0   1.0551
H5-Z5                     1.1511                    1.2        181.9229         167.2805                   1.0   1.0875

OK: round on Implied/Realized, truncate on Cap vol Impl/Rlzd, 4/13 finite rows.
The two rules give a DIFFERENT printed value on 3 of 4 of them, so the asymmetry is not cosmetic.


## 3. The sign probe

Re-derived from the data on every run rather than trusted from a comment.
Two things have to be true for the screen to mean what it says:

1. `ca_bp` is `(pack rate − matched swap rate)` in bp, so a **positive** CA
   means the futures strip prices above the swap. Selling the adjustment is
   therefore *buy the four futures, pay fixed on the matched swap* — the Citi
   quote at the top of this notebook.
2. Every ranking metric points the **same way**: higher = more attractive
   short-convexity candidate. `rank_flags` is one `nlargest` per metric, so
   raising a pack's metric must be able to flag it and lowering it must not.

In [5]:
_today_rows = GATED[GATED["date"] == AS_OF]
_id = (_today_rows["pack_rate"] - _today_rows["swap_rate"]) * 100.0
_err = float((_id - _today_rows["ca_bp"]).abs().max())
print(f"CA identity  max |(pack_rate - swap_rate)*100 - ca_bp| = {_err:.3e} bp "
      f"over {len(_today_rows)} rows")
assert _err < 1e-6, "ca_bp is not the pack-minus-swap spread in bp"
print(f"sign: {int((_today_rows['ca_bp'] > 0).sum())}/{len(_today_rows)} packs have CA > 0, "
      "i.e. the strip prices ABOVE the matched swap; selling that adjustment is")
print("      BUY the four SR3 contracts and PAY fixed on the matched-maturity swap.")

CA identity  max |(pack_rate - swap_rate)*100 - ca_bp| = 0.000e+00 bp over 16 rows
sign: 16/16 packs have CA > 0, i.e. the strip prices ABOVE the matched swap; selling that adjustment is
      BUY the four SR3 contracts and PAY fixed on the matched-maturity swap.


The monotonicity probe. Built on the live screen, which is computed in
section 6; this cell is deferred until after it and re-appears there as an
assert. Here we state the rule so the reader meets it before the numbers.

## 4. The date, and how stale it is

The screen can only be as current as the panel behind it. Both the panel's
last **gated** date and the wall clock at execution are printed, because the
gap between them is the thing a live screener has to disclose.

In [6]:
_all_dates = pd.DatetimeIndex(sorted(panel_all["date"].unique()))
_gated_dates = pd.DatetimeIndex(sorted(GATED["date"].unique()))
STALE_BD = int(np.busday_count(AS_OF.date(), RUN_AT.date()))

print(f"panel file          {PANEL_PATH.name}  "
      f"(mtime {dt.datetime.fromtimestamp(PANEL_PATH.stat().st_mtime):%Y-%m-%d %H:%M})")
print(f"panel dates         {len(_all_dates):,}   {_all_dates.min():%Y-%m-%d} .. {_all_dates.max():%Y-%m-%d}")
print(f"gate-passing dates  {len(_gated_dates):,}   {_gated_dates.min():%Y-%m-%d} .. {_gated_dates.max():%Y-%m-%d}")
print()
print(f"  ===>  SCREEN AS OF   {AS_OF:%Y-%m-%d} ({AS_OF.day_name()})")
print(f"  ===>  EXECUTED       {RUN_AT:%Y-%m-%d}")
print(f"  ===>  STALENESS      {STALE_BD} business day(s)")

assert AS_OF in _gated_dates, "the chosen date is not in the gated panel"
assert AS_OF == _gated_dates.max(), "not the most recent available date"
if AS_OF != _all_dates.max():
    print(f"\nWARNING: the panel's newest date is {_all_dates.max():%Y-%m-%d} and it "
          "fails the gate entirely, so the newest data is being discarded and the "
          "screen is older than the file's date range suggests.")
if STALE_BD > 5:
    print(f"\nWARNING: the panel is {STALE_BD} business days behind the execution date. "
          "Re-warm strat2_q20_panel.parquet before treating this as live.")
else:
    print(f"\nOK: the screen is the most recent gate-passing date and is {STALE_BD} "
          "business day(s) behind execution.")

panel file          strat2_q20_panel.parquet  (mtime 2026-08-21 01:50)
panel dates         2,070   2018-05-04 .. 2026-08-20
gate-passing dates  2,070   2018-05-04 .. 2026-08-20

  ===>  SCREEN AS OF   2026-08-20 (Thursday)
  ===>  EXECUTED       2026-08-21
  ===>  STALENESS      1 business day(s)

OK: the screen is the most recent gate-passing date and is 1 business day(s) behind execution.


## 5. Building the screen

`panel_timeseries` and `model_timeseries` are computed once over the whole
config window and handed to `daily_screen`, so the z-scores and the fitted
sigma see the full history rather than being recomputed per call.

In [7]:
SUB = GATED[(GATED["date"] >= pd.Timestamp(CONFIG.start))
            & (GATED["date"] <= pd.Timestamp(CONFIG.end))].copy()
TS = S2.panel_timeseries(SUB, CONFIG)
MODEL = S2.model_timeseries(SUB, CONFIG)
print(f"screen window {SUB['date'].nunique():,} dates "
      f"{SUB['date'].min():%Y-%m-%d} .. {SUB['date'].max():%Y-%m-%d}, {len(SUB):,} rows")
print(f"ts keys    {list(TS)}")
print(f"model keys {list(MODEL)}")

SCREEN = S2.daily_screen(AS_OF.date(), SUB, CONFIG, ts=TS, model=MODEL)
assert not SCREEN.empty, "the live screen is EMPTY — there is nothing to act on"
print(f"\nscreen {SCREEN.shape[0]} rows x {SCREEN.shape[1]} columns")

screen window 1,406 dates 2021-01-04 .. 2026-08-20, 22,608 rows
ts keys    ['ca', 'pack_rate', 'rv', 'ca_z3m', 'ca_z1y', 'ca_chg_1w']
model keys ['fit', 'vs_model', 'sigma_model', 'ca_model', 'vs_model_z3m', 'vs_model_z1y']

screen 15 rows x 20 columns


## 6. Citi's 13-column daily screen, as of the date above

The published column order, from the `daily_screen` docstring:

```
CvxAdj | 1WkChg | 3m ZS | 1Y ZS | Model(bp) | Vs Model(bp) | 3m ZS |
1Y ZS | 3m Roll | Implied Vol | Realized Vol | Implied/Realized |
Cap vol Impl/Rlzd
```

In [8]:
PUBLISHED = ["ca_bp", "ca_chg_1w_bp", "ca_z3m", "ca_z1y",
             "ca_model_bp", "vs_model_bp", "vs_model_z3m", "vs_model_z1y",
             "roll_3m_bp", "implied_vol_bp", "realized_vol_bp",
             "implied_over_realized", "capvol_over_realized"]
HEADERS = ["CvxAdj", "1WkChg", "3m ZS", "1Y ZS", "Model(bp)", "Vs Model(bp)",
           "3m ZS", "1Y ZS", "3m Roll", "Implied Vol", "Realized Vol",
           "Implied/Realized", "Cap vol Impl/Rlzd"]
assert len(PUBLISHED) == len(HEADERS) == 13
_pos = [list(SCREEN.columns).index(c) for c in PUBLISHED]
assert _pos == sorted(_pos), (
    f"the 13 published columns are not in Citi's order in the frame: {_pos}")
print("OK: all 13 published columns present, in the published relative order.")

TABLE = SCREEN[["rank", "colour", "swap_start", "swap_end"] + PUBLISHED].copy()
print(f"\nCONVEXITY ADJUSTMENTS FOR 1Y SOFR PACKS — {AS_OF:%d-%b-%Y}\n")
print(TABLE.round(3).to_string())

OK: all 13 published columns present, in the published relative order.

CONVEXITY ADJUSTMENTS FOR 1Y SOFR PACKS — 20-Aug-2026

       rank  colour  swap_start    swap_end   ca_bp  ca_chg_1w_bp  ca_z3m  ca_z1y  ca_model_bp  vs_model_bp  vs_model_z3m  vs_model_z1y  roll_3m_bp  implied_vol_bp  realized_vol_bp  implied_over_realized  capvol_over_realized
pack                                                                                                                                                                                                                             
Z6-U7     2    None  2026-12-16  2027-12-15   0.850         0.734   0.345   0.688        0.104        0.746         0.766         1.022       0.129         173.650           80.475                    2.2                   0.7
H7-Z7     3    None  2027-03-17  2028-03-15   1.132         0.682   0.364   0.782        0.238        0.894         0.849         1.211       0.282         152.463           82.655               

In [9]:
# the monotonicity half of the sign probe, on the live screen (section 3)
_probes = []
for m in CONFIG.rank_metrics:
    s = SCREEN[m].dropna()
    if len(s) < CONFIG.top_n_per_metric + 1:
        continue
    base = S2.rank_flags(SCREEN, CONFIG)
    top3 = set(s.nlargest(CONFIG.top_n_per_metric).index)
    assert set(base.index[base[m]]) == top3, f"{m}: flags are not the top {CONFIG.top_n_per_metric}"
    victim = s.idxmin()
    up = SCREEN.copy(); up.loc[victim, m] = s.max() + abs(s.max()) + 1.0
    dn = SCREEN.copy(); dn.loc[victim, m] = s.min() - abs(s.min()) - 1.0
    f_up = bool(S2.rank_flags(up, CONFIG).at[victim, m])
    f_dn = bool(S2.rank_flags(dn, CONFIG).at[victim, m])
    _probes.append({"metric": m, "n_finite": len(s), "victim": victim,
                    "flagged_when_raised": f_up, "flagged_when_lowered": f_dn})
    assert f_up and not f_dn, f"{m} does not rank HIGHER = more attractive short"
print(pd.DataFrame(_probes).to_string(index=False))
print(f"\nOK: all {len(_probes)} rankable metrics point the same way — higher is a more")
print("attractive SHORT-convexity candidate, and the flags are exactly the top 3.")

               metric  n_finite victim  flagged_when_raised  flagged_when_lowered
                ca_bp        15  Z6-U7                 True                 False
               ca_z3m        10  U7-M8                 True                 False
               ca_z1y         6  H9-Z9                 True                 False
          vs_model_bp        15  M8-H9                 True                 False
         vs_model_z3m        10  M9-H0                 True                 False
         vs_model_z1y         6  H9-Z9                 True                 False
           roll_3m_bp        14  U7-M8                 True                 False
implied_over_realized        10  U7-M8                 True                 False

OK: all 8 rankable metrics point the same way — higher is a more
attractive SHORT-convexity candidate, and the flags are exactly the top 3.


## 7. What the rule would trade now

`rank_flags` marks the three best packs per metric; `select_pack` takes the
pack flagged by the most, breaking ties on mean cross-sectional percentile.
With `min_metrics_flagged=0` the rule **never stays flat** — Citi rolled the
theme from Blues to Greens rather than closing it, and this config reproduces
that.

In [10]:
FLAGS = S2.rank_flags(SCREEN, CONFIG)
PICK, FL = S2.select_pack(SCREEN, CONFIG)
ORDERED = FL.sort_values(["n_flags", "mean_pct"], ascending=False)
print(ORDERED.to_string())

assert PICK is not None, "select_pack returned nothing on a non-empty screen"
assert PICK in SCREEN.index, "the selected pack is not in the screen"
_row = SCREEN.loc[PICK]
_ties = ORDERED[ORDERED["n_flags"] == ORDERED["n_flags"].max()]
print(f"\n  ===>  SELECTED PACK   {PICK}   rank {int(_row['rank'])}"
      f"   colour {_row['colour'] or '(unnamed rank)'}")
print(f"        flagged by       {int(FL.at[PICK, 'n_flags'])} of "
      f"{len(CONFIG.rank_metrics)} metrics: "
      f"{[m for m in CONFIG.rank_metrics if bool(FLAGS.at[PICK, m])]}")
print(f"        matched swap     {_row['swap_start']} .. {_row['swap_end']}")
print(f"        CA               {_row['ca_bp']:.2f}bp   vs model "
      f"{_row['vs_model_bp']:+.2f}bp   3m roll {_row['roll_3m_bp']:+.2f}bp")
if len(_ties) > 1:
    _tie_txt = ", ".join("%s %.3f" % (p, ORDERED.at[p, "mean_pct"]) for p in _ties.index)
    print(f"\n        TIE on n_flags among {list(_ties.index)}; broken on mean_pct "
          f"({_tie_txt}).")
    print("        The tiebreak is deterministic but it is a tiebreak, not a signal.")

       ca_bp  ca_z3m  ca_z1y  vs_model_bp  vs_model_z3m  vs_model_z1y  roll_3m_bp  implied_over_realized  n_flags  mean_pct
pack                                                                                                                       
H7-Z7  False   False    True         True          True          True       False                   True        5  0.706548
M7-H8  False   False    True         True          True          True       False                   True        5  0.561607
Z6-U7  False   False    True         True         False          True       False                   True        4  0.589286
M0-H1   True    True   False        False         False         False        True                  False        3  0.662302
U9-M0  False   False   False        False          True         False        True                  False        2  0.785317
H0-Z0   True    True   False        False         False         False       False                  False        2  0.626587
Z9-U0  F

In [11]:
_r = SCREEN.sort_values("rank")
fig = go.Figure()
fig.add_trace(go.Scatter(x=_r["rank"], y=_r["ca_bp"], name="CA (bp)",
                         mode="lines+markers"))
fig.add_trace(go.Scatter(x=_r["rank"], y=_r["ca_model_bp"], name="fitted model (bp)",
                         mode="lines", line=dict(dash="dash")))
_f3 = _r[_r.index.isin(FL[FL["n_flags"] >= 3].index)]
if len(_f3):
    fig.add_trace(go.Scatter(x=_f3["rank"], y=_f3["ca_bp"], name="flagged by >=3 metrics",
                             mode="markers", marker=dict(size=13, symbol="circle-open")))
fig.add_trace(go.Scatter(x=[_row["rank"]], y=[_row["ca_bp"]], name=f"SELECTED {PICK}",
                         mode="markers", marker=dict(size=16, symbol="x")))
fig.update_layout(title=f"SOFR 1y pack convexity adjustment, {AS_OF:%d-%b-%Y}",
                  xaxis_title="pack rank (1 = front)", yaxis_title="bp",
                  height=420)
fig.show()

## 8. The caveat that lives inside the flag count

`rank_flags` can only flag a metric it has a finite value for. Today the
metrics are **not uniformly populated across the strip**, so packs with a
fuller z-score history can accumulate more flags than packs with an equally
extreme but unmeasurable one. This is a coverage artefact, not a signal, and
it is printed rather than left to be discovered.

In [12]:
FINITE = SCREEN[list(CONFIG.rank_metrics)].notna().sum(axis=1).rename("n_finite")
BIAS = pd.concat([SCREEN["rank"], FINITE, FL["n_flags"], FL["mean_pct"]], axis=1)
print(BIAS.sort_values("rank").to_string())
print("\nfinite values per metric across the strip:")
print(SCREEN[list(CONFIG.rank_metrics)].notna().sum().to_string())

_full = SCREEN[FINITE == len(CONFIG.rank_metrics)]
_alt, _ = S2.select_pack(_full, CONFIG) if len(_full) else (None, None)
print(f"\npacks with ALL {len(CONFIG.rank_metrics)} metrics finite: {list(_full.index)}")
print(f"restricted to those, the pick would be {_alt} (actual pick {PICK}) — "
      f"{'UNCHANGED' if _alt == PICK else 'DIFFERENT, so the flag count is coverage-driven'}")
print("\nThis is context, not an alternative rule. The traded rule is select_pack on the")
print("full screen; the restriction is only here to show whether coverage moved it.")

       rank  n_finite  n_flags  mean_pct
pack                                    
Z6-U7     2         8        4  0.589286
H7-Z7     3         8        5  0.706548
M7-H8     4         8        5  0.561607
U7-M8     5         8        0  0.288095
H8-Z8     7         2        0  0.166667
M8-H9     8         3        0  0.274603
U8-M9     9         3        0  0.484127
Z8-U9    10         3        0  0.457143
H9-Z9    11         8        0  0.414286
M9-H0    12         8        1  0.560417
U9-M0    13         6        2  0.785317
Z9-U0    14         6        1  0.629762
H0-Z0    15         6        2  0.626587
M0-H1    16         6        3  0.662302
U0-M1    17         3        1  0.611111

finite values per metric across the strip:
ca_bp                    15
ca_z3m                   10
ca_z1y                    6
vs_model_bp              15
vs_model_z3m             10
vs_model_z1y              6
roll_3m_bp               14
implied_over_realized    10

packs with ALL 8 metrics finite: [

In [13]:
# the missing rank, and which gate component rejected it
_ranks_today = set(SCREEN["rank"])
_expected = set(range(CONFIG.rank_start, CONFIG.rank_start + CONFIG.n_packs))
_missing = sorted(_expected - _ranks_today)
print(f"ranks requested {min(_expected)}..{max(_expected)}, present {len(_ranks_today)}, "
      f"missing {_missing}")
if _missing:
    _u = panel_all[(panel_all["date"] == AS_OF) & (panel_all["rank"].isin(_missing))]
    print("\nthe ungated panel rows for the missing rank(s):")
    print(_u[["rank", "pack", "gate_covered", "gate_resolved", "gate_settle_agrees",
              "gate_ok", "max_settle_diff_bp", "ca_bp"]].to_string(index=False))
    for _, _q in _u.iterrows():
        _why = [c for c in ("gate_covered", "gate_resolved", "gate_settle_agrees")
                if not bool(_q[c])]
        _hist_r = panel_all[panel_all["rank"] == _q["rank"]].sort_values("date").tail(20)
        _passed = int(_hist_r["gate_ok"].sum())
        _shape = "an intermittent" if _passed >= 10 else "a persistent"
        print(f"\nrank {int(_q['rank'])} ({_q['pack']}) fails {_why}; over the last "
              f"{len(_hist_r)} panel dates for that rank it passed on {_passed} — "
              f"{_shape} hole.")
    print("\nA missing rank silently shortens the cross-section the model is fitted to")
    print("and removes a candidate from the ranking. It is reported, not repaired here.")
else:
    print("no missing ranks on this date.")

ranks requested 2..17, present 15, missing [6]

the ungated panel rows for the missing rank(s):
 rank  pack  gate_covered  gate_resolved  gate_settle_agrees  gate_ok  max_settle_diff_bp   ca_bp
    6 Z7-U8          True          False                True    False            0.565566 0.84636

rank 6 (Z7-U8) fails ['gate_resolved']; over the last 20 panel dates for that rank it passed on 18 — an intermittent hole.

A missing rank silently shortens the cross-section the model is fitted to
and removes a candidate from the ranking. It is reported, not repaired here.


## 9. Enrichment — dealer positioning, open interest, CCP basis

From the `ca_signals` module docstring, quoting Citi's own explanation of what
widens the adjustment:

> *"Dealers, who are on the other side of the shorts established by hedge funds
> and asset managers, have ended up with significant long ED positions
> (Figure 2). Convexity adjustments have therefore widened to compensate
> dealers for this concentration risk."*

The **provenance dict is printed in full** because two of these columns are
routinely mistaken for something they are not. In particular `open_interest`
here is the CFTC **whole-strip** aggregate for SOFR-3M — it has no
per-contract dimension, and it is not the open interest of the selected pack.

In [14]:
BASIS, BASIS_NOTE = None, ""
try:
    from MDP.IRClearingHouseBasisSwaps.ccp_basis_cache import basis_panel
    BASIS = basis_panel(CONFIG.start, AS_OF.date(),
                        tenors=list(CS.EnrichmentConfig().basis_tenors),
                        allow_network=False)
    BASIS_NOTE = (f"{BASIS.shape[0]} dates {BASIS.index.min():%Y-%m-%d}.."
                  f"{BASIS.index.max():%Y-%m-%d}, {BASIS.attrs.get('sign_convention')}, "
                  f"units {BASIS.attrs.get('units')}")
    print(f"CCP basis cache: {BASIS_NOTE}")
    print(BASIS.tail(3).to_string())
except Exception as exc:                                          # noqa: BLE001
    BASIS, BASIS_NOTE = None, f"{type(exc).__name__}: {exc}"
    print(f"CCP basis ABSENT — {BASIS_NOTE}")
    print("(reported absent rather than substituted; nothing downstream fabricates it)")

ECFG = CS.EnrichmentConfig(start=CONFIG.start, end=AS_OF.date())
ENR, PROV = CS.build_enrichment_panel(ECFG, basis=BASIS)
print(f"\nenrichment panel {ENR.shape}, {ENR.index.min():%Y-%m-%d}..{ENR.index.max():%Y-%m-%d}")
print("\nPROVENANCE — which source served each column, and at what lag:")
for k, v in PROV.items():
    print(f"  {k:20} {v}")

CCP basis cache: 1413 dates 2021-01-04..2026-08-20, LCH minus CME, units bp
             5y   10y  30y
date                      
2026-08-18 -0.6 -2.05 -2.3
2026-08-19 -0.6 -2.00 -2.2
2026-08-20 -0.6 -2.00 -2.2



enrichment panel (1470, 9), 2021-01-01..2026-08-20

PROVENANCE — which source served each column, and at what lag:
  dealer_net           CFTC TFF SOFR3M, weekly, lagged 3 bdays to publication
  lev_net              CFTC TFF SOFR3M, weekly, lagged 3 bdays to publication
  am_net               CFTC TFF SOFR3M, weekly, lagged 3 bdays to publication
  dealer_net_chg       first difference of the lagged level
  open_interest        CFTC TFF Open_Interest_All for SOFR3M — WHOLE STRIP, weekly, no per-contract dimension. Per-contract SR3 OI is survivorship-shaped on this machine and does not exist for front ranks before ~2025.
  oi_chg               first difference of the lagged level
  ccp_basis_5y_bp      LCH minus CME, USD SOFR 5y, lagged 1 bday
  ccp_basis_10y_bp     LCH minus CME, USD SOFR 10y, lagged 1 bday
  ccp_basis_30y_bp     LCH minus CME, USD SOFR 30y, lagged 1 bday


### 9.1 What is actually attached to today's screen

The provenance strings say the *lag rule*. They do not say when the underlying
series last actually moved, which on a weekly report is the number that
matters. Both are printed.

In [15]:
ENRICHED = CS.enrich_screen(SCREEN, ENR, AS_OF.date())
ENR_COLS = [c for c in ENRICHED.columns if c.startswith("enr_")]
print(f"attached columns: {ENR_COLS}")

REQUIRED = ["dealer_net", "lev_net", "am_net", "open_interest"]
_missing_enr = [c for c in REQUIRED if f"enr_{c}" not in ENRICHED.columns]
assert not _missing_enr, f"required enrichment columns absent: {_missing_enr}"
_basis_cols = [c for c in ENR_COLS if c.startswith("enr_ccp_basis")]
if _basis_cols:
    print(f"CCP basis columns PRESENT: {_basis_cols}")
else:
    print(f"CCP basis columns ABSENT — reason: {BASIS_NOTE or 'no cache supplied'}")

_freshness = []
for c in ENR.columns:
    s = ENR[c].dropna()
    if s.empty:
        _freshness.append({"column": c, "value": np.nan, "last_change": None, "days_stale": np.nan})
        continue
    chg = s[s.diff().fillna(1.0) != 0.0]
    last_chg = chg.index[-1] if len(chg) else s.index[-1]
    _freshness.append({"column": c, "value": float(s.iloc[-1]),
                       "last_change": last_chg.date(),
                       "days_stale": int((AS_OF - last_chg).days)})
FRESH = pd.DataFrame(_freshness).set_index("column")
print("\nthe row attached to EVERY pack of today's screen, and how old it is:")
print(FRESH.to_string())

_ds = int(FRESH.at["dealer_net", "days_stale"])
print(f"\nNOTE: dealer_net last changed {_ds} calendar days before the screen date, "
      "measured on")
print("the publication-LAGGED index; the underlying report is 3 business days older")
print("still (section 10). The CFTC report is weekly, so a few days is normal and a")
print("hundred is not. The CCP basis, by contrast, is a market observable and is current.")
assert ENRICHED[[f"enr_{c}" for c in REQUIRED]].notna().all().all(), (
    "an enrichment column attached as all-NaN, which is absence dressed as data")

attached columns: ['enr_dealer_net', 'enr_lev_net', 'enr_am_net', 'enr_dealer_net_chg', 'enr_open_interest', 'enr_oi_chg', 'enr_ccp_basis_5y_bp', 'enr_ccp_basis_10y_bp', 'enr_ccp_basis_30y_bp']
CCP basis columns PRESENT: ['enr_ccp_basis_5y_bp', 'enr_ccp_basis_10y_bp', 'enr_ccp_basis_30y_bp']

the row attached to EVERY pack of today's screen, and how old it is:
                       value last_change  days_stale
column                                              
dealer_net         1719008.0  2026-05-15          97
lev_net           -1530754.0  2026-05-15          97
am_net             -244061.0  2026-05-15          97
dealer_net_chg           0.0  2026-05-18          94
open_interest     13320834.0  2026-05-15          97
oi_chg                   0.0  2026-05-18          94
ccp_basis_5y_bp         -0.6  2026-08-20           0
ccp_basis_10y_bp        -2.0  2026-08-20           0
ccp_basis_30y_bp        -2.2  2026-08-20           0

NOTE: dealer_net last changed 97 calendar days before

## 10. The current dealer positioning, in historical context

Citi's thesis is that a **concentrated dealer long** is what widens the
adjustment. That makes an extreme positioning reading the single most
important output of this screen — and the sample says the current reading is
not merely extreme, it is the maximum.

Three claims are re-derived here rather than quoted: that dealers sit on the
other side of leveraged money and asset managers; the sample extremes; and
where today's reading falls in that sample.

In [16]:
HCFG = CS.EnrichmentConfig(start=dt.date(2020, 1, 1), end=dt.date(2026, 5, 31))
try:
    from BT.signals.cftc_positioning import build_positioning_panel, fetch_cftc_financial_futures
    _cache = REPO / "BT" / "results" / "tfp_screener" / "cftc_raw.parquet"
    _raw = fetch_cftc_financial_futures(cache_path=str(_cache))
    HIST = pd.DataFrame({m: build_positioning_panel(raw=_raw, tenors=[HCFG.cftc_market],
                                                    metric=m)[HCFG.cftc_market]
                         for m in ("dealer_net", "lev_net", "am_net")}).dropna()
    _raw_last = pd.to_datetime(_raw["Report_Date_as_YYYY-MM-DD"]).max()
except Exception as exc:                                          # noqa: BLE001
    print(f"raw CFTC path unavailable ({type(exc).__name__}: {exc}); "
          "falling back to the enrichment panel")
    HIST, _ = CS.build_enrichment_panel(HCFG)
    HIST = HIST[["dealer_net", "lev_net", "am_net"]].dropna()
    _raw_last = None

HIST = HIST[HIST.index <= pd.Timestamp("2026-05-31")]
OTHER = HIST["lev_net"] + HIST["am_net"]
CORR = float(HIST["dealer_net"].corr(OTHER))
print(f"sample {len(HIST):,} rows  {HIST.index.min():%Y-%m-%d} .. {HIST.index.max():%Y-%m-%d}")
print(f"corr(dealer_net, leveraged + asset manager) in LEVELS = {CORR:.4f}")
assert -0.99 < CORR < -0.90, (
    f"the 'other side of the shorts' mechanism is not visible: corr {CORR:.4f}")

DEALER_MAX, LEV_MIN = float(HIST["dealer_net"].max()), float(HIST["lev_net"].min())
print(f"\ndealer_net  max {DEALER_MAX:>14,.0f}  on {HIST['dealer_net'].idxmax():%Y-%m-%d}"
      f"   min {HIST['dealer_net'].min():>14,.0f}")
print(f"lev_net     min {LEV_MIN:>14,.0f}  on {HIST['lev_net'].idxmin():%Y-%m-%d}"
      f"   max {HIST['lev_net'].max():>14,.0f}")
assert abs(DEALER_MAX - 1_719_008) < 1.0, (
    f"the recorded dealer maximum of 1,719,008 no longer reproduces: {DEALER_MAX:,.0f}")
assert abs(LEV_MIN + 1_530_754) < 1.0, (
    f"the recorded leveraged minimum of -1,530,754 no longer reproduces: {LEV_MIN:,.0f}")
print("\nOK: the two recorded extremes reproduce exactly, and the -0.95 level "
      "correlation holds.")

sample 1,656 rows  2020-01-10 .. 2026-05-15
corr(dealer_net, leveraged + asset manager) in LEVELS = -0.9547

dealer_net  max      1,719,008  on 2026-05-15   min     -1,118,131
lev_net     min     -1,530,754  on 2026-05-15   max      1,422,541

OK: the two recorded extremes reproduce exactly, and the -0.95 level correlation holds.


In [17]:
CUR_DEALER = float(ENRICHED[f"enr_dealer_net"].iloc[0])
CUR_LEV = float(ENRICHED[f"enr_lev_net"].iloc[0])
CUR_AM = float(ENRICHED[f"enr_am_net"].iloc[0])
PCTL = float((HIST["dealer_net"] <= CUR_DEALER).mean())
LEV_PCTL = float((HIST["lev_net"] <= CUR_LEV).mean())
print(f"reading carried onto today's screen (ffilled from the last report):")
print(f"  dealer_net  {CUR_DEALER:>14,.0f}   percentile of the 2020-2026 sample  {PCTL:.3f}")
print(f"  lev_net     {CUR_LEV:>14,.0f}   percentile                            {LEV_PCTL:.3f}")
print(f"  am_net      {CUR_AM:>14,.0f}")
print(f"  sum of the three reported categories {CUR_DEALER + CUR_LEV + CUR_AM:>14,.0f}  "
      "(not zero: other reportables and")
print("                                                     non-reportables are not "
      "in these three)")

assert PCTL >= 0.99, f"dealer positioning is no longer at an extreme ({PCTL:.3f})"
print("\n  ===>  DEALERS ARE AT THE MAXIMUM LONG OF THE SAMPLE, and leveraged money at")
print("        its minimum. On Citi's mechanism that is the configuration in which the")
print("        convexity adjustment is WIDE and is expected to stay wide.")

if _raw_last is not None:
    _lagd = int((AS_OF - _raw_last).days)
    print(f"\n  ===>  AND IT IS OLD. The CFTC cache ON THIS MACHINE ends at report date")
    print(f"        {_raw_last:%Y-%m-%d}, {_lagd} calendar days ({_lagd/7:.0f} weeks) before the screen")
    print("        date. This notebook cannot see whether newer reports exist upstream;")
    print("        what it can say is that the local cache does not have them. The")
    print("        'currently long' reading above is a 12-May-2026 observation carried")
    print("        forward, not a current one.")
    assert _lagd >= 0, "the positioning report post-dates the screen — a look-ahead"

reading carried onto today's screen (ffilled from the last report):
  dealer_net       1,719,008   percentile of the 2020-2026 sample  1.000
  lev_net         -1,530,754   percentile                            0.001
  am_net            -244,061
  sum of the three reported categories        -55,807  (not zero: other reportables and
                                                     non-reportables are not in these three)

  ===>  DEALERS ARE AT THE MAXIMUM LONG OF THE SAMPLE, and leveraged money at
        its minimum. On Citi's mechanism that is the configuration in which the
        convexity adjustment is WIDE and is expected to stay wide.

  ===>  AND IT IS OLD. The CFTC cache ON THIS MACHINE ends at report date
        2026-05-12, 100 calendar days (14 weeks) before the screen
        date. This notebook cannot see whether newer reports exist upstream;
        what it can say is that the local cache does not have them. The
        'currently long' reading above is a 12-May-2026 o

In [18]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=HIST.index, y=HIST["dealer_net"], name="dealer net"))
fig.add_trace(go.Scatter(x=HIST.index, y=OTHER, name="leveraged + asset manager"))
fig.add_trace(go.Scatter(x=[HIST.index[-1]], y=[float(HIST["dealer_net"].iloc[-1])],
                         name="last available reading (carried onto today's screen)",
                         mode="markers", marker=dict(size=14, symbol="x")))
fig.update_layout(
    title=(f"CFTC TFF SOFR-3M net positioning (lagged 3 bdays to publication) — "
           f"corr {CORR:.4f} in levels"),
    yaxis_title="contracts, net", height=420)
fig.show()

## 11. What the backtest of this exact rule says, and what the rule has picked

Section 7 produces a trade. Section 11 is what constrains how much weight it
deserves: the same config, run over 2021–2026 by `_w2b_ca_vs_fly_run.py`.

In [19]:
ARMS_CSV = DATA / "w2b_arms.csv"
ARMS = None
try:
    if ARMS_CSV.exists():
        ARMS = pd.read_csv(ARMS_CSV)
        assert {"arm", "terminal", "sharpe"} <= set(ARMS.columns)
        print(f"read {ARMS_CSV.name} "
              f"(mtime {dt.datetime.fromtimestamp(ARMS_CSV.stat().st_mtime):%Y-%m-%d %H:%M})")
except Exception as exc:                                          # noqa: BLE001
    print(f"w2b_arms.csv unreadable ({type(exc).__name__}: {exc})")
    ARMS = None

if ARMS is None:
    rows = []
    for f in sorted(DATA.glob("w2b_equity_*.parquet")):
        e = pd.read_parquet(f)["equity"]
        e.index = pd.to_datetime(e.index)
        r = e.diff().dropna()
        rows.append({"arm": f.stem.replace("w2b_equity_", ""),
                     "terminal": float(e.iloc[-1]),
                     "sharpe": float(r.mean() / r.std() * np.sqrt(252)) if r.std() > 0 else np.nan,
                     "max_dd": float((e - e.cummax()).min()),
                     "span_years": (e.index[-1] - e.index[0]).days / 365.25})
    ARMS = pd.DataFrame(rows)
    print(f"w2b_arms.csv absent — recomputed {len(ARMS)} arm(s) from the equity parquets")

NET_SHARPE, BREAKEVEN_BP = np.nan, np.nan
if len(ARMS):
    print("\n" + ARMS.round(4).to_string(index=False))
    _u0 = ARMS[ARMS["arm"] == "unhedged_zero_cost"]
    _u5 = ARMS[ARMS["arm"] == "unhedged_base"]
    if len(_u0) and len(_u5):
        g, n = float(_u0["terminal"].iloc[0]), float(_u5["terminal"].iloc[0])
        BREAKEVEN_BP = 0.5 * g / (g - n) if g != n else np.nan
        NET_SHARPE = float(_u5["sharpe"].iloc[0])
        print(f"\ngross (0 cost)  {g:>14,.0f}   net (0.5bp round trip) {n:>14,.0f}")
        print(f"cost of the book at 0.5bp: {g - n:,.0f}  ({(g - n) / g:.1%} of gross)")
        print(f"break-even round-trip cost: {BREAKEVEN_BP:.3f} bp")
        print(f"net annualised Sharpe at 0.5bp: {NET_SHARPE:.3f}")
        assert abs(NET_SHARPE) < 0.35, (
            "the W2b net Sharpe is no longer ~zero; section 12's verdict needs re-deriving")
        _pairs = [(f"unhedged_{t}", f"hedged_{t}")
                  for t in ("zero_cost", "base", "cost_1bp")]
        _worse = []
        for _un, _he in _pairs:
            _a, _b = ARMS[ARMS["arm"] == _un], ARMS[ARMS["arm"] == _he]
            if len(_a) and len(_b):
                _worse.append({"cost arm": _un.replace("unhedged_", ""),
                               "unhedged": float(_a["terminal"].iloc[0]),
                               "hedged": float(_b["terminal"].iloc[0]),
                               "hedge helps": float(_b["terminal"].iloc[0])
                                              > float(_a["terminal"].iloc[0])})
        if _worse:
            print("\nthe 2s5s10s hedge, arm by arm:")
            print(pd.DataFrame(_worse).to_string(index=False))
            assert not any(w["hedge helps"] for w in _worse), (
                "the hedged arm now beats the unhedged one somewhere; section 12's "
                "'strictly worse' claim needs re-deriving")
        print("\nVERDICT CARRIED FORWARD: the rule this screen feeds made approximately")
        print("nothing net of a 0.5bp round trip, is negative at 1bp, and the 2s5s10s")
        print("hedge made it worse rather than better. The screen is a valuation")
        print("statement; it is not, on this evidence, a profitable instruction.")
    else:
        print("\nthe zero-cost / 0.5bp arm pair is not present; no break-even derivable")
else:
    print("NO W2b ARTIFACTS FOUND — the backtest verdict is UNAVAILABLE and section 12")
    print("cannot be weighed against it. Run _w2b_ca_vs_fly_run.py.")

read w2b_arms.csv (mtime 2026-08-21 05:34)

               arm  hedged  cost_bp      terminal  sharpe        max_dd  span_years  n_epochs
unhedged_zero_cost   False      0.0  2.483251e+06  0.1296 -2.620061e+06      5.6235        45
     unhedged_base   False      0.5  2.332513e+05  0.0122 -2.870061e+06      5.6235        45
 unhedged_cost_1bp   False      1.0 -2.016749e+06 -0.1049 -3.607512e+06      5.6235        45
  hedged_zero_cost    True      0.0  2.395284e+06  0.1254 -2.659882e+06      5.6235        45
       hedged_base    True      0.5  1.452836e+05  0.0076 -2.909882e+06      5.6235        45
   hedged_cost_1bp    True      1.0 -2.104716e+06 -0.1097 -3.625304e+06      5.6235        45

gross (0 cost)       2,483,251   net (0.5bp round trip)        233,251
cost of the book at 0.5bp: 2,250,000  (90.6% of gross)
break-even round-trip cost: 0.552 bp
net annualised Sharpe at 0.5bp: 0.012

the 2s5s10s hedge, arm by arm:
 cost arm      unhedged        hedged  hedge helps
zero_cost  2.

In [20]:
# what the rule has actually picked, on its own BMS rebalance dates
_bms = pd.date_range(CONFIG.start, AS_OF, freq=CONFIG.rebalance_freq)
_days = pd.DatetimeIndex(sorted(SUB["date"].unique()))
_picks = []
for _d in _bms:
    _dd = _days[_days <= _d]
    if not len(_dd):
        continue
    _s = S2.daily_screen(_dd[-1].date(), SUB, CONFIG, ts=TS, model=MODEL)
    if _s.empty:
        continue
    _p, _f = S2.select_pack(_s, CONFIG)
    if _p is None:
        continue
    _picks.append({"date": _dd[-1], "pack": _p, "rank": int(_s.at[_p, "rank"]),
                   "colour": _s.at[_p, "colour"], "n_flags": int(_f.at[_p, "n_flags"])})
PICKS = pd.DataFrame(_picks)
print(f"{len(PICKS)} monthly rebalance dates, {int((PICKS['pack'] != PICKS['pack'].shift()).sum())} "
      "changes of pack")
print("\nrank of the selected pack, distribution:")
print(PICKS["rank"].value_counts().sort_index().to_string())
FRONT_SHARE = float((PICKS["rank"] <= 4).mean())
DEEP_SHARE = float((PICKS["rank"] >= 13).mean())
print(f"\nrank <= 4 (front of the strip): {FRONT_SHARE:.1%} of dates")
print(f"rank >= 13 (Blues and beyond) : {DEEP_SHARE:.1%} of dates")
_where = ("the FRONT of the strip, not the deferred Blues concentration Citi wrote about"
          if int(_row["rank"]) <= 4 else "in the deferred part of the strip")
print(f"\ntoday's pick is rank {int(_row['rank'])}, which is {_where}.")
print("\nlast 12 rebalance picks:")
print(PICKS.tail(12).to_string(index=False))

assert len(PICKS) > 24, "too few rebalance dates to characterise what the rule picks"

67 monthly rebalance dates, 56 changes of pack

rank of the selected pack, distribution:
rank
2     3
3     5
4     6
5     8
6     5
7     7
8     3
9     5
10    5
11    2
12    4
13    3
15    2
16    6
17    3

rank <= 4 (front of the strip): 20.9% of dates
rank >= 13 (Blues and beyond) : 20.9% of dates

today's pick is rank 3, which is the FRONT of the strip, not the deferred Blues concentration Citi wrote about.

last 12 rebalance picks:
      date  pack  rank colour  n_flags
2025-08-29 M8-H9    12   None        4
2025-10-01 U8-M9    12   None        4
2025-11-03 M6-H7     3   None        4
2025-12-01 M8-H9    11   None        5
2025-12-31 M8-H9    10   None        6
2026-02-02 H7-Z7     5   Reds        4
2026-03-02 U9-M0    15   None        4
2026-04-01 H9-Z9    12   None        5
2026-05-01 M7-H8     5   Reds        5
2026-06-01 M9-H0    13  Blues        5
2026-07-01 Z6-U7     2   None        5
2026-08-03 M0-H1    16   None        4


## 12. What this says to do today

The instruction, the evidence for it, and the reasons to discount it — in that
order, all of them printed from the cells above rather than written here.

In [21]:
_flagged = [m for m in CONFIG.rank_metrics if bool(FLAGS.at[PICK, m])]
print("=" * 78)
print(f"SCREEN DATE {AS_OF:%d-%b-%Y}   (executed {RUN_AT:%d-%b-%Y}, "
      f"{STALE_BD} business day(s) later)")
print("=" * 78)
print()
print("THE TRADE THE RULE SELECTS")
print(f"  Sell ${CONFIG.ca_dv01:,.0f} DV01 of the {PICK} convexity adjustment:")
print(f"    BUY the four SR3 contracts of the {PICK} pack, and")
print(f"    PAY fixed on the matched-maturity {_row['swap_start']} .. {_row['swap_end']} swap.")
print(f"  Rank {int(_row['rank'])} of {CONFIG.rank_start}..{CONFIG.rank_start + CONFIG.n_packs - 1}"
      f"   colour {_row['colour'] or '(unnamed rank)'}")
print(f"  CA {_row['ca_bp']:.2f}bp, model {_row['ca_model_bp']:.2f}bp, "
      f"dislocation {_row['vs_model_bp']:+.2f}bp")
print(f"  Flagged by {len(_flagged)}/{len(CONFIG.rank_metrics)} metrics: {_flagged}")
print(f"  Config hedge_enabled={CONFIG.hedge_enabled} would add the 2s5s10s fly; the")
print("  backtest in section 11 measured the hedged arm as STRICTLY WORSE.")
print()
print("THE EVIDENCE FOR IT")
print(f"  Dealer net positioning is at the sample MAXIMUM ({CUR_DEALER:,.0f} contracts,")
print(f"  percentile {PCTL:.3f} of 2020-2026), leveraged money at its minimum")
print(f"  ({CUR_LEV:,.0f}), the two correlated {CORR:.4f} in levels. On Citi's stated")
print("  mechanism that is precisely the concentration that keeps adjustments wide.")
print()
print("THE CAVEATS, EACH MEASURED ABOVE")
if _raw_last is not None:
    print(f"  1. That positioning reading is from the {_raw_last:%d-%b-%Y} report — "
          f"{int((AS_OF - _raw_last).days)} days")
    print("     stale, carried forward by ffill. It is the newest the local cache holds.")
else:
    print("  1. The positioning provenance could not be dated on this run.")
if np.isfinite(NET_SHARPE):
    print(f"  2. The backtested rule returned a net Sharpe of {NET_SHARPE:.3f} at a 0.5bp")
    print(f"     round trip, breaks even at {BREAKEVEN_BP:.2f}bp, and is negative at 1bp.")
    print("     The screen is not the edge; the screen is where to look.")
else:
    print("  2. The W2b backtest artifacts were unavailable on this run, so the screen")
    print("     is UNCONSTRAINED here. That is a gap, not a licence.")
print(f"  3. Today's pick sits at rank {int(_row['rank'])}. Historically the rule picks the front")
print(f"     four ranks {FRONT_SHARE:.0%} of the time, so a front pick is normal — but it is")
print("     NOT the deferred Blues concentration that Citi's positioning thesis is about.")
print(f"  4. Metric coverage is uneven: {int(FINITE.max())}/{len(CONFIG.rank_metrics)} metrics finite at best, "
      f"{int(FINITE.min())} at worst")
print(f"     across the {len(SCREEN)} packs, and rank(s) {_missing or 'none'} are missing from the")
print("     screen entirely. Flags are only awarded on finite metrics.")
print()
print("=" * 78)

SCREEN DATE 20-Aug-2026   (executed 21-Aug-2026, 1 business day(s) later)

THE TRADE THE RULE SELECTS
  Sell $100,000 DV01 of the H7-Z7 convexity adjustment:
    BUY the four SR3 contracts of the H7-Z7 pack, and
    PAY fixed on the matched-maturity 2027-03-17 .. 2028-03-15 swap.
  Rank 3 of 2..17   colour (unnamed rank)
  CA 1.13bp, model 0.24bp, dislocation +0.89bp
  Flagged by 5/8 metrics: ['ca_z1y', 'vs_model_bp', 'vs_model_z3m', 'vs_model_z1y', 'implied_over_realized']
  Config hedge_enabled=True would add the 2s5s10s fly; the
  backtest in section 11 measured the hedged arm as STRICTLY WORSE.

THE EVIDENCE FOR IT
  Dealer net positioning is at the sample MAXIMUM (1,719,008 contracts,
  percentile 1.000 of 2020-2026), leveraged money at its minimum
  (-1,530,754), the two correlated -0.9547 in levels. On Citi's stated
  mechanism that is precisely the concentration that keeps adjustments wide.

THE CAVEATS, EACH MEASURED ABOVE
  1. That positioning reading is from the 12-May-2026 

## Reading this notebook

* **The screen reproduces where it can be graded, and fails where the module
  said it would.** Section 2 grades 13 published Citi rows: the convexity
  adjustment itself reproduces in level and rank, the fitted model reproduces
  in shape, and `vs_model` does not reproduce at all — which is the documented
  consequence of fitting sigma to the same cross-section it is compared
  against. Anyone reading `Vs Model (bp)` here as Citi's dislocation is
  reading the wrong number.

* **The positioning extreme is the screener's most important output, and it is
  also its stalest input.** Both halves are true simultaneously. Dealers at
  the sample maximum long is exactly the configuration Citi's mechanism
  predicts a wide adjustment from; the reading is a report from months before
  the screen date, held constant since. Treat it as a regime statement, not a
  current observation.

* **`open_interest` is not per contract.** It is the CFTC whole-strip
  aggregate for SOFR-3M, broadcast onto every pack row. The provenance dict in
  section 9 says so in words; the `enr_` prefix on the column says so in the
  frame. Per-contract SR3 open interest on this machine is survivorship-shaped
  and does not exist for front ranks before ~2025, so there is nothing better
  to attach.

* **The flag count is part signal and part coverage.** Section 8 prints the
  finite-metric count next to the flag count for every pack. A pack cannot be
  flagged on a metric it has no finite value for, so the packs with the
  longest clean z-score history are structurally advantaged. On this date the
  restriction to fully-covered packs does not move the pick, but that is a
  measurement for this date and not a property of the rule.

* **What would have to change for the screen to be actionable.** Not the
  screen — the trade. Section 11's backtest of this exact config clears
  approximately nothing net of a 0.5bp round trip, breaks even a little above
  that, and is worse hedged than unhedged. The screen is worth running because
  it locates the dislocation; the evidence that the dislocation is harvestable
  at this size, cadence and cost is what is missing.